In [ ]:
!apt-get install -y tesseract-ocr
!pip install pytesseract opencv-python numpy


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  tesseract-ocr-eng tesseract-ocr-osd
The following NEW packages will be installed:
  tesseract-ocr tesseract-ocr-eng tesseract-ocr-osd
0 upgraded, 3 newly installed, 0 to remove and 20 not upgraded.
Need to get 4,816 kB of archives.
After this operation, 15.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-eng all 1:4.00~git30-7274cfa-1.1 [1,591 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-osd all 1:4.00~git30-7274cfa-1.1 [2,990 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr amd64 4.1.1-2.1build1 [236 kB]
Fetched 4,816 kB in 1s (3,666 kB/s)
Selecting previously unselected package tesseract-ocr-eng.
(Reading database ... 124926 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving CS124S5.pdf to CS124S5.pdf


In [ ]:
!apt-get install -y poppler-utils
!pip install pdf2image


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 20 not upgraded.
Need to get 186 kB of archives.
After this operation, 696 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.6 [186 kB]
Fetched 186 kB in 1s (272 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 124973 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.6_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.6) ...
Setting up poppler-utils (22.02.0-2ubuntu0.6) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
import cv2
import numpy as np
import pytesseract
import os
import time
from pdf2image import convert_from_path
import google.generativeai as genai

# API Key 
GOOGLE_API_KEY = ""
genai.configure(api_key=GOOGLE_API_KEY)


pdf_path = "/content/CS124S5.pdf"
images = convert_from_path(pdf_path, dpi=300)

full_text = ""  

# Extract OCR 
for page_num, image in enumerate(images):
    image = np.array(image)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    
    custom_config = r'--oem 3 --psm 6'
    text = pytesseract.image_to_string(gray, config=custom_config)

    full_text += f"\n--- Page {page_num + 1} ---\n" + text

# prompt for Gemini
prompt = f"""
You are an AI assistant. The following is a scanned question paper.
Your task is to accurately extract and structure questions, options,
and answers if available. Example output in plain text format:

Q1. What is AI?
   A) A programming language
   B) A robot
   C) Machine learning
   D) None of the above
   Answer: C  (Only if available)

Q2. ...

Here is the extracted text:
{full_text}
"""

retries = 0
response_text = None

while retries < max_retries:
    try:
        response = genai.GenerativeModel("gemini-pro").generate_content(prompt)
        if response and hasattr(response, "text"):  
            response_text = response.text
        break  
    except Exception as e:
        if "429" in str(e):  
            retries += 1
            wait_time = 2 ** retries  
            print(f"Rate limit exceeded. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            print(f"Error calling Gemini API: {e}")
            break  


if response_text:
    txt_filename = "structured_questions.txt"
    with open(txt_filename, "w", encoding="utf-8") as f:
        f.write(response_text)
    print(f"Questions and options saved to {txt_filename}")
else:
    print("No structured text received from Gemini API.")


Questions and options saved to structured_questions.txt


In [ ]:
import google.generativeai as genai
import time

GOOGLE_API_KEY = ""
genai.configure(api_key=GOOGLE_API_KEY)


txt_filename = "structured_questions.txt"
with open(txt_filename, "r", encoding="utf-8") as f:
    questions_data = f.read()

# Process each question
questions_list = questions_data.strip().split("\n\n")
enriched_data = ""

for question_block in questions_list:
    if not question_block.strip():
        continue  

   
    prompt = f"""
    You are an AI tutor. For the given question, determine:
    - The subject, course, and topic it belongs to.
    - Verify the correct answer.
    - Provide a short detailed explanation.

    Question Block:
    {question_block}

    Respond in this format:
    {question_block}
    Subject: <subject>
    Course: <course>
    Topic: <topic>
    Correct Answer: <option>
    Explanation: <brief detailed answer>
    """

    
    try:
        response = genai.GenerativeModel("gemini-pro").generate_content(prompt)
        enriched_data += f"{response.text}\n\n"
    except Exception as e:
        print(f"Error processing question:\n{question_block}\n⚠️ Error: {e}\n")


    time.sleep(2)


output_filename = "enriched_questions.txt"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(enriched_data)

print(f"Enriched questions saved to {output_filename}")


Enriched questions saved to enriched_questions.txt
